# RoadSafe Delhi — Logistic Regression Model Development
## Assigned Member: Amit
### Task: Parametric Probabilistic Road Risk Classification & Log-Odds Analysis

---

## 1. Project Introduction & Assigned Member
- **Project:** RoadSafe Delhi — Explainable Road Risk Mapping and Safety Intervention Prioritization Using Machine Learning
- **Assigned Member:** Amit
- **Model Assignment:** Logistic Regression (`sklearn.linear_model.LogisticRegression`)
- **Role:** Logistic Regression model development, odds-ratio interpretability, decision boundary calibration, and literature research (5 papers).


## 2. Problem Statement & Model Objective
The objective is to train a calibrated regularized Logistic Regression classifier to estimate the log-odds of crash occurrence across Delhi road segments:
$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \sum_{j=1}^p \beta_j X_j$$
- **Interpretability:** Exponentiated coefficients $\exp(\beta_j)$ provide direct odds ratios for municipal transport safety policies.


## 3. Dataset Loading from Common Approved Dataset
Loading the shared spatial partitions from `data_final/`.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
DATA_FINAL_DIR = os.path.join(PROJECT_ROOT, "data_final")
ARTIFACTS_DIR = os.path.join(PROJECT_ROOT, "models", "logistic_regression", "artifacts")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "models", "logistic_regression", "outputs")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

df_train = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "train.parquet"))
df_val = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "validation.parquet"))
df_test = pd.read_parquet(os.path.join(DATA_FINAL_DIR, "test.parquet"))

print(f"Loaded Train: {len(df_train):,}, Val: {len(df_val):,}, Test: {len(df_test):,}")


## 4. Target Variable Selection
Selected Target: `is_crash_segment` (Binary Hazard Occurrence).


In [ ]:
target_col = "is_crash_segment"
print(f"Target: {target_col}")


## 5. Approved Feature Selection
Loading `feature_contract.json` and asserting zero target leakage.


In [ ]:
with open(os.path.join(DATA_FINAL_DIR, "feature_contract.json"), "r") as f:
    contract = json.load(f)

num_features = contract["approved_numerical_predictors"]
cat_features = contract["approved_categorical_predictors"]
approved_predictors = num_features + cat_features

forbidden = set(contract["quarantined_leakage_columns"]).union(set(contract["target_definitions"].keys()))
assert len(set(approved_predictors).intersection(forbidden)) == 0, "FATAL: Leakage detected!"
print("Approved Predictors:", approved_predictors)


## 6. Data Preprocessing (Standard Scaling & One-Hot Encoding)
Fitting transformations strictly on the training partition.


In [ ]:
scaler = StandardScaler()
scaler.fit(df_train[num_features])

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(df_train[cat_features])

X_train = np.hstack([scaler.transform(df_train[num_features]), encoder.transform(df_train[cat_features])])
X_val = np.hstack([scaler.transform(df_val[num_features]), encoder.transform(df_val[cat_features])])
X_test = np.hstack([scaler.transform(df_test[num_features]), encoder.transform(df_test[cat_features])])

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values

feature_names = num_features + encoder.get_feature_names_out(cat_features).tolist()
print(f"Prepared {X_train.shape[1]} features for Logistic Regression.")


## 7. Train-Validation-Test Data Verification (Common Spatial Split)
Verifying disjoint spatial block boundaries.


In [ ]:
assert set(df_train['spatial_block']).isdisjoint(set(df_val['spatial_block']))
assert set(df_train['spatial_block']).isdisjoint(set(df_test['spatial_block']))
print("Spatial Independence Confirmed.")


## 8. Model Architecture & Configuration
Configuring `LogisticRegression` with L2 penalty, class weighting, and lbfgs solver.


In [ ]:
log_model = LogisticRegression(
    C=1.0,
    penalty="l2",
    class_weight="balanced",
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)
print("Configured Logistic Regression Model:", log_model)


## 9. Model Training
Fitting model on `X_train`, `y_train`.


In [ ]:
log_model.fit(X_train, y_train)
print("Logistic Regression Training Completed.")


## 10. Validation & Hyperparameter / Threshold Tuning
Tuning decision threshold strictly on the validation set to maximize F1-score.


In [ ]:
from sklearn.metrics import precision_recall_curve

y_val_prob = log_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)
f1_scores = np.zeros_like(thresholds)
for i, t in enumerate(thresholds):
    f1_scores[i] = f1_score(y_val, (y_val_prob >= t).astype(int), zero_division=0)

best_idx = np.argmax(f1_scores)
optimal_threshold = float(thresholds[best_idx])
print(f"Optimal Validation Threshold: {optimal_threshold:.4f} (Validation F1 = {f1_scores[best_idx]:.4f})")


## 11. Final Testing on the Untouched Test Dataset
Evaluating model on `X_test` using the frozen validation threshold.


In [ ]:
y_test_prob = log_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)

print(f"Test Accuracy:  {accuracy_score(y_test, y_test_pred)*100:.2f}%")
print(f"Test Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test Recall:    {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test F1-Score:  {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"Test PR-AUC:    {average_precision_score(y_test, y_test_prob):.4f}")


## 12. Evaluation Metrics & Visualizations
Plotting Odds Ratios and Confusion Matrix.


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", ax=ax)
ax.set_title(f"Logistic Regression Test Confusion Matrix (Threshold={optimal_threshold:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "confusion_matrix.png"), dpi=200)
plt.show()

# Odds Ratios
odds_ratios = pd.Series(np.exp(log_model.coef_[0]), index=feature_names).sort_values()
plt.figure(figsize=(10, 6))
odds_ratios.plot(kind="barh", color="#d35400")
plt.axvline(1.0, color="gray", linestyle="--")
plt.title("Logistic Regression Odds Ratios exp(β)")
plt.xlabel("Odds Ratio (>1 increases risk odds, <1 decreases)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "odds_ratios.png"), dpi=200)
plt.show()


## 13. Results Interpretation
Analysis of significant risk multipliers and policy implications.


In [ ]:
print("Top Risk Multipliers (Odds Ratio > 1):\n", odds_ratios.tail(4))
print("\nTop Mitigating Factors (Odds Ratio < 1):\n", odds_ratios.head(4))


## 14. Model and Artifact Saving
Saving model, scaler, encoder, threshold, and metrics.


In [ ]:
import joblib

joblib.dump(log_model, os.path.join(ARTIFACTS_DIR, "logistic_regression_model.joblib"))
joblib.dump(scaler, os.path.join(ARTIFACTS_DIR, "scaler.joblib"))
joblib.dump(encoder, os.path.join(ARTIFACTS_DIR, "encoder.joblib"))
with open(os.path.join(ARTIFACTS_DIR, "threshold.json"), "w") as f:
    json.dump({"optimal_threshold": optimal_threshold}, f, indent=2)

print(f"Saved Logistic Regression artifacts to {ARTIFACTS_DIR}")


## 15. Final Summary & Observations
Summary of parametric odds-ratio findings and comparison with other architectures.
